In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Wazirpur_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,373.0,233.0,233.0,NaN,215.0,264.0,100.0,72.0,106.0,163.0,373.0,298.0
1,2,359.0,284.0,153.0,150.0,194.0,158.0,127.0,98.0,104.0,217.0,344.0,293.0
2,3,359.0,278.0,148.0,182.0,286.0,142.0,162.0,126.0,160.0,192.0,420.0,300.0
3,4,416.0,342.0,159.0,173.0,286.0,210.0,100.0,114.0,84.0,242.0,417.0,172.0
4,5,364.0,267.0,157.0,195.0,340.0,222.0,130.0,89.0,101.0,177.0,426.0,168.0
5,6,350.0,194.0,134.0,188.0,251.0,151.0,98.0,NaN,139.0,188.0,426.0,211.0
6,7,384.0,224.0,199.0,186.0,328.0,196.0,92.0,83.0,129.0,152.0,435.0,255.0
7,8,394.0,213.0,171.0,215.0,249.0,207.0,90.0,96.0,108.0,188.0,416.0,370.0
8,9,369.0,169.0,144.0,197.0,188.0,143.0,105.0,119.0,150.0,187.0,386.0,255.0
9,10,315.0,391.0,184.0,230.0,192.0,138.0,205.0,96.0,152.0,148.0,369.0,266.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,373.0,233.000000,233.000000,174.617647,215.000000,155.441176,100.000000,72.000000,106.000000,163.000000,373.000000,298.000000
1,2,359.0,284.000000,153.000000,150.000000,194.000000,158.000000,127.000000,98.000000,104.000000,217.000000,344.000000,293.000000
2,3,359.0,278.000000,148.000000,182.000000,286.000000,142.000000,162.000000,126.000000,160.000000,192.000000,420.000000,300.000000
3,4,416.0,342.000000,159.000000,173.000000,286.000000,210.000000,100.000000,114.000000,84.000000,242.000000,417.000000,172.000000
4,5,364.0,267.000000,157.000000,195.000000,212.028571,222.000000,130.000000,89.000000,101.000000,177.000000,426.000000,168.000000
5,6,350.0,194.000000,134.000000,188.000000,251.000000,151.000000,98.000000,92.575758,139.000000,188.000000,426.000000,211.000000
6,7,384.0,224.000000,199.000000,186.000000,328.000000,196.000000,92.000000,83.000000,129.000000,152.000000,435.000000,255.000000
7,8,394.0,213.000000,171.000000,215.000000,249.000000,207.000000,90.000000,96.000000,108.000000,188.000000,416.000000,370.000000
8,9,369.0,169.000000,144.000000,197.000000,188.000000,143.000000,105.000000,119.000000,150.000000,187.000000,386.000000,255.000000
9,10,315.0,391.000000,184.000000,230.000000,192.000000,138.000000,110.735294,96.000000,152.000000,148.000000,369.000000,266.000000
